# Xử lý và chia dữ liệu HetRec Last.fm 2K

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/trongnguyen1010/musicrec-collaborative-filtering/blob/docs/notebook-xu-ly-du-lieu/notebooks/xu_ly_va_chia_du_lieu.ipynb)

Làm sạch → chia 80/10/10 theo từng người dùng → biểu diễn đầu vào, từ tệp gốc của GroupLens.
Các hàm xử lý được chép nguyên từ `src/musicrec/data/cleaning.py`, `splitting.py` và `representations.py`.

Chạy: **Runtime → Run all** (khoảng 1 phút, không cần GPU).

## 0. Thư viện và tham số

In [ ]:
import csv
import hashlib
import math
import random
import re
import urllib.request
import zipfile
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from scipy.sparse import csr_matrix

# Tham số lấy từ configs/dataset.yaml và configs/base.yaml
SOURCE_URL = "https://files.grouplens.org/datasets/hetrec2011/hetrec2011-lastfm-2k.zip"
SOURCE_SHA256 = "6738f48195667ff03caaab4d32ca9a3133d8cc026b7c3cdaf6ce1010e913c59c"
MIN_USER_INTERACTIONS = 5
SEED = 42
VALIDATION_FRACTION = 0.1
TEST_FRACTION = 0.1


def so(value, decimals=0):
    """Viết số theo kiểu báo cáo: dấu chấm ngăn hàng nghìn, dấu phẩy thập phân (1.892; 0,2783)."""
    return f"{value:,.{decimals}f}".replace(",", " ").replace(".", ",").replace(" ", ".")

## 1. Tải dữ liệu gốc

In [ ]:
zip_path = Path("hetrec2011-lastfm-2k.zip")
if not zip_path.exists():
    urllib.request.urlretrieve(SOURCE_URL, zip_path)
print("SHA-256:", hashlib.sha256(zip_path.read_bytes()).hexdigest())
print("Cấu hình:", SOURCE_SHA256)

RAW = Path("hetrec2011")
with zipfile.ZipFile(zip_path) as archive:
    archive.extractall(RAW)
print(sorted(p.name for p in RAW.iterdir()))

### Xem dữ liệu gốc

`user_artists.dat`: ai nghe nghệ sĩ nào, bao nhiêu lượt (dữ liệu chính). `artists.dat`: danh mục nghệ sĩ.
`tags.dat`: từ điển thẻ. `user_taggedartists.dat`: người dùng gắn thẻ cho nghệ sĩ.
`weight` là **tổng lượt nghe**, không phải điểm đánh giá (phản hồi ngầm).

In [ ]:
def read_source_table(path: Path) -> pd.DataFrame:
    """Đọc tệp .dat phân tách bằng TAB, giữ nguyên chữ "NA" và bảng mã của từng tệp."""
    frame = pd.read_csv(
        path,
        sep="\t",
        encoding="latin-1" if path.name == "tags.dat" else "utf-8",
        dtype=str,
        keep_default_na=False,
        quoting=csv.QUOTE_NONE,
    )
    return frame


raw_user_artists = read_source_table(RAW / "user_artists.dat")
raw_artists = read_source_table(RAW / "artists.dat")
raw_tags = read_source_table(RAW / "tags.dat")
raw_assignments = read_source_table(RAW / "user_taggedartists.dat")
for name, frame in [("user_artists.dat", raw_user_artists), ("artists.dat", raw_artists),
                    ("tags.dat", raw_tags), ("user_taggedartists.dat", raw_assignments)]:
    print(f"{name}: {so(frame.shape[0])} dòng × {frame.shape[1]} cột")
    display(frame.head())

In [ ]:
# Thống kê mô tả (Bảng 3.4); hàm này dùng lại cho dữ liệu sau lọc
N_CATALOG = len(raw_artists)

def describe(frame, user, item, count):
    per_user = frame.groupby(user)[item].nunique()
    per_item = frame.groupby(item)[user].nunique()
    listens = frame[count].astype("int64")
    n_users = frame[user].nunique()
    exactly_50 = (per_user == 50)
    # Viết sẵn thành chuỗi giống Bảng 3.4 để bảng không bị chuyển sang dạng 1.892000e+03
    return {
        "Số người dùng": so(n_users),
        "Số nghệ sĩ trong danh mục": so(N_CATALOG),
        "Số nghệ sĩ có tương tác": so(frame[item].nunique()),
        "Số cặp người dùng–nghệ sĩ": so(len(frame)),
        "Mật độ ma trận (trên danh mục)": so(100 * len(frame) / (n_users * N_CATALOG), 4) + "%",
        "Số nghệ sĩ mỗi người dùng: trung bình / trung vị": f"{so(per_user.mean(), 2)} / {so(per_user.median())}",
        "Số nghệ sĩ mỗi người dùng: nhỏ nhất / lớn nhất": f"{so(per_user.min())} / {so(per_user.max())}",
        "Số người dùng có đúng 50 nghệ sĩ": f"{so(exactly_50.sum())} ({so(100 * exactly_50.mean(), 2)}%)",
        "Số người nghe mỗi nghệ sĩ có tương tác: trung bình / trung vị":
            f"{so(per_item.mean(), 2)} / {so(per_item.median())}",
        "Số nghệ sĩ chỉ có một người nghe": so((per_item == 1).sum()),
        "Tổng số lượt nghe": so(listens.sum()),
        "Số lượt nghe của một cặp: trung vị / lớn nhất": f"{so(listens.median())} / {so(listens.max())}",
    }

raw_stats = describe(raw_user_artists, "userID", "artistID", "weight")
display(pd.Series(raw_stats, name="Dữ liệu gốc").to_frame())

outside_catalog = ~raw_assignments.artistID.isin(set(raw_artists.id))
listeners = raw_user_artists.groupby("artistID").userID.nunique()
few_listeners = set(listeners[listeners < 5].index)
print("Số vị trí của ma trận:", so(raw_user_artists.userID.nunique() * N_CATALOG))
print("Liên kết ảnh trống:", so(raw_artists.pictureURL.str.strip().eq("").sum()))
print("Dòng gán thẻ:", so(len(raw_assignments)), "| nối được danh mục:", so((~outside_catalog).sum()),
      "| không nối được:", so(outside_catalog.sum()),
      "thuộc", so(raw_assignments.loc[outside_catalog, "artistID"].nunique()), "mã nghệ sĩ")
print("Nếu lọc nghệ sĩ dưới 5 người nghe:", so(len(few_listeners)), "nghệ sĩ,",
      so(raw_user_artists.artistID.isin(few_listeners).sum()), "tương tác bị ảnh hưởng (không áp dụng)")
print("Thẻ ngoài ASCII:", raw_tags.tagValue[~raw_tags.tagValue.map(str.isascii)].tolist())

## 2. Làm sạch

Kiểm tra ID, lượt nghe, nghệ sĩ ngoài danh mục, cặp trùng; loại người dùng có ít hơn 5 nghệ sĩ.
Không lọc nghệ sĩ, không biến đổi lượt nghe.

In [ ]:
def clean_interactions(
    source: pd.DataFrame, artist_ids: set[str], min_user_interactions: int = 5
) -> tuple[pd.DataFrame, pd.DataFrame, dict[str, int]]:
    """Làm sạch bảng user_artists: loại dòng lỗi, cặp trùng và người dùng có dưới 5 nghệ sĩ.

    Nhận bảng gốc và danh mục nghệ sĩ; trả (bảng sạch, bảng dòng bị loại kèm lý do, tóm tắt).
    Cặp trùng có lượt nghe khác nhau bị báo lỗi thay vì cộng dồn, vì không biết giá trị nào đúng.
    """
    if type(min_user_interactions) is not int or min_user_interactions < 1:
        raise ValueError("min_user_interactions must be a positive integer")
    frame = source.loc[:, ["userID", "artistID", "weight"]].copy()
    frame["source_line"] = range(2, len(frame) + 2)
    frame["removal_reason"] = ""
    valid_ids = (
        frame[["userID", "artistID"]]
        .apply(lambda column: column.str.fullmatch(r"[1-9][0-9]*", na=False))
        .all(axis=1)
    )
    # Lượt nghe trong HetRec là số nguyên dương int64, không phải điểm đánh giá hay số thực.
    integers = frame.weight.str.fullmatch(r"[0-9]+", na=False)
    counts = frame.weight.where(integers, "0").map(int)
    valid_counts = integers & counts.gt(0) & counts.le(2**63 - 1)
    for invalid, reason in [
        (~valid_ids, "invalid_id"),
        (~valid_counts, "invalid_count"),
        (~frame.artistID.isin(artist_ids), "artist_outside_catalog"),
    ]:
        frame.loc[invalid & frame.removal_reason.eq(""), "removal_reason"] = reason
    eligible = frame.removal_reason.eq("")
    observed = frame.loc[eligible, ["userID", "artistID"]].assign(weight=counts[eligible])
    conflicts = observed.groupby(["userID", "artistID"]).weight.nunique().gt(1)
    if conflicts.any():
        raise ValueError(
            f"Conflicting counts for user-artist pairs: {list(conflicts[conflicts].index)}"
        )
    duplicate = observed.duplicated(["userID", "artistID", "weight"])
    frame.loc[observed.index[duplicate], "removal_reason"] = "duplicate_observation"
    eligible = frame.removal_reason.eq("")
    # Người có dưới min_user_interactions (5) nghệ sĩ không đủ để chia ba tập huấn luyện,
    # xác thực, kiểm thử nên bị loại.
    histories = frame.loc[eligible].groupby("userID").artistID.nunique()
    short_users = set(histories[histories < min_user_interactions].index)
    frame.loc[eligible & frame.userID.isin(short_users), "removal_reason"] = "user_below_minimum"
    kept = frame.loc[frame.removal_reason.eq(""), ["userID", "artistID"]].copy()
    kept["listening_count"] = counts.loc[kept.index].astype("int64")
    kept = kept.rename(columns={"userID": "user_id", "artistID": "item_id"})
    kept = kept.sort_values(["user_id", "item_id"]).reset_index(drop=True)
    excluded = frame.loc[frame.removal_reason.ne("")].reset_index(drop=True)
    reasons = [
        "invalid_id",
        "invalid_count",
        "artist_outside_catalog",
        "duplicate_observation",
        "user_below_minimum",
    ]
    summary = {
        "input_rows": len(frame),
        "output_rows": len(kept),
        "excluded_rows": len(excluded),
        "users_below_minimum": len(short_users),
        **{reason: int(frame.removal_reason.eq(reason).sum()) for reason in reasons},
    }
    return kept, excluded, summary


def clean_tables(
    interactions: pd.DataFrame,
    artists: pd.DataFrame,
    tags: pd.DataFrame,
    assignments: pd.DataFrame,
    min_user_interactions: int = 5,
) -> tuple[dict[str, pd.DataFrame], dict[str, Any]]:
    """Làm sạch các bảng nghệ sĩ, thẻ và gán thẻ; tách thẻ của nghệ sĩ ngoài danh mục ra riêng."""
    for frame, key in [(artists, "id"), (tags, "tagID")]:
        if (
            frame[key].duplicated().any()
            or not frame[key].str.fullmatch(r"[1-9][0-9]*", na=False).all()
        ):
            raise ValueError(f"Invalid or duplicate catalog key: {key}")
    artist_ids = set(artists.id)
    cleaned, excluded, quality = clean_interactions(interactions, artist_ids, min_user_interactions)
    if cleaned.empty:
        raise ValueError("Cleaning leaves no interactions")
    for key, allowed in [("userID", set(interactions.userID)), ("tagID", set(tags.tagID))]:
        if not assignments[key].isin(allowed).all():
            raise ValueError(f"Tag assignments contain unknown {key}")
    outside = ~assignments.artistID.isin(artist_ids)
    orphan = assignments.loc[outside].copy()
    orphan["source_line"] = assignments.index[outside] + 2
    orphan["removal_reason"] = "artist_outside_catalog"
    valid_tags = assignments.loc[~outside].copy()
    artists = artists.copy()
    artists["display_name"] = artists.name.where(
        artists.name.str.strip().ne(""), "Artist " + artists.id
    )
    tagged_ids, active_ids = set(valid_tags.artistID), set(cleaned.item_id)
    tables = {
        "interactions": cleaned,
        "excluded_interactions": excluded,
        "artists": artists,
        "tags": tags.copy(),
        "tag_assignments": valid_tags,
        "excluded_tag_assignments": orphan,
    }
    metrics = {
        "interaction_rules": quality,
        "before": {
            "users": int(interactions.userID.nunique()),
            "interaction_artists": int(interactions.artistID.nunique()),
            "interactions": len(interactions),
            "catalog_artists": len(artists),
        },
        "after": {
            "users": int(cleaned.user_id.nunique()),
            "interaction_artists": len(active_ids),
            "interactions": len(cleaned),
            "catalog_artists": len(artists),
            "total_listens": int(cleaned.listening_count.sum()),
            "min_user_history": int(cleaned.groupby("user_id").item_id.nunique().min()),
            "single_listener_artists": int(
                cleaned.groupby("item_id").user_id.nunique().eq(1).sum()
            ),
            "max_listening_count": int(cleaned.listening_count.max()),
        },
        "tags": {
            "dictionary_rows": len(tags),
            "input_assignments": len(assignments),
            "retained_assignments": len(valid_tags),
            "excluded_assignments": len(orphan),
            "excluded_artist_ids": int(orphan.artistID.nunique()),
            "catalog_artists_with_tags": len(artist_ids & tagged_ids),
            "catalog_artists_without_tags": len(artist_ids - tagged_ids),
            "active_artists_without_tags": len(active_ids - tagged_ids),
            "assignments_by_excluded_users_retained_for_later_protocol_filtering": int(
                (~valid_tags.userID.isin(set(cleaned.user_id))).sum()
            ),
            "scope": "Source metadata only; retain userID for later M2 protocol filtering",
        },
    }
    return tables, metrics

In [ ]:
clean, clean_metrics = clean_tables(
    raw_user_artists, raw_artists, raw_tags, raw_assignments, min_user_interactions=MIN_USER_INTERACTIONS
)
labels = {"users": "Người dùng", "interaction_artists": "Nghệ sĩ có tương tác",
          "interactions": "Cặp người dùng–nghệ sĩ", "catalog_artists": "Nghệ sĩ trong danh mục"}
display(pd.DataFrame({"Trước": {labels[k]: so(clean_metrics["before"][k]) for k in labels},
                      "Sau": {labels[k]: so(clean_metrics["after"][k]) for k in labels}}))
rules = {"input_rows": "Dòng đầu vào", "output_rows": "Dòng giữ lại", "excluded_rows": "Dòng bị loại",
         "invalid_id": "Bị loại do mã không hợp lệ", "invalid_count": "Bị loại do lượt nghe không hợp lệ",
         "artist_outside_catalog": "Bị loại do nghệ sĩ ngoài danh mục", "duplicate_observation": "Bị loại do trùng cặp",
         "user_below_minimum": "Bị loại do người dùng dưới 5 nghệ sĩ", "users_below_minimum": "Số người dùng bị loại"}
display(pd.Series({rules[k]: so(clean_metrics["interaction_rules"][k]) for k in rules}, name="Số lượng").to_frame())

In [ ]:
# Những dòng bị loại, kèm số dòng trong tệp gốc user_artists.dat
excluded = clean["excluded_interactions"]
print(so(len(excluded)), "dòng bị loại,", so(excluded.userID.nunique()), "người dùng")
display(excluded)

In [ ]:
interactions = clean["interactions"]  # đầu vào của bước chia
display(interactions.head(10))
display(clean["tag_assignments"].head())
tag_labels = {
    "dictionary_rows": "Số thẻ trong từ điển",
    "input_assignments": "Dòng gán thẻ đầu vào",
    "retained_assignments": "Dòng gán thẻ giữ lại (nối được danh mục)",
    "excluded_assignments": "Dòng gán thẻ lưu riêng (không nối được)",
    "excluded_artist_ids": "Số mã nghệ sĩ của các dòng lưu riêng",
    "catalog_artists_with_tags": "Nghệ sĩ trong danh mục có thẻ",
    "catalog_artists_without_tags": "Nghệ sĩ trong danh mục chưa có thẻ",
    "active_artists_without_tags": "Nghệ sĩ chưa có thẻ nhưng có tương tác sau lọc",
    "assignments_by_excluded_users_retained_for_later_protocol_filtering": "Dòng gán thẻ của người dùng đã bị loại",
}
display(pd.Series({tag_labels[k]: so(clean_metrics["tags"][k]) for k in tag_labels}, name="Thẻ").to_frame())

In [ ]:
# Bảng 3.4: dữ liệu gốc và sau khi lọc
after_stats = describe(interactions, "user_id", "item_id", "listening_count")
display(pd.DataFrame({"Dữ liệu gốc": raw_stats, "Sau khi lọc (≥ 5 nghệ sĩ/người)": after_stats}))
print("Tổng lượt nghe ba tập sau khi chia sẽ phải bằng", after_stats["Tổng số lượt nghe"])

## 3. Chia 80/10/10 theo từng người dùng

Công thức (3.2): `n_val = n_test = max(1, ⌊0,1·n⌋)`, phần còn lại vào train; xáo trộn bằng `random.Random(42)`.
Ánh xạ ID chỉ tạo từ train; đích ngoài danh mục train vẫn được giữ, `item_index` rỗng.

In [ ]:
def split_interactions(
    interactions: pd.DataFrame,
    seed: int = 42,
    validation_fraction: float = 0.1,
    test_fraction: float = 0.1,
) -> tuple[dict[str, pd.DataFrame], dict[str, dict[str, int]], dict[str, Any]]:
    """Chia 80/10/10 trong từng người dùng với hạt giống cố định, công thức (3.2).

    Nhận bảng đã làm sạch; trả ba tập, ánh xạ mã → chỉ số (chỉ từ tập huấn luyện) và
    thống kê danh mục. Dữ liệu không có dấu thời gian nên chia ngẫu nhiên trong từng người.
    """
    columns = ["user_id", "item_id", "listening_count"]
    if list(interactions.columns) != columns or interactions.empty:
        raise ValueError("Expected non-empty cleaned user_id/item_id/listening_count columns")
    frame = interactions.copy()
    if frame.isna().any().any():
        raise ValueError("Cleaned interactions must not contain missing values")
    for column in ["user_id", "item_id"]:
        if (
            not frame[column]
            .map(
                lambda value: (
                    isinstance(value, str) and re.fullmatch(r"[1-9][0-9]*", value) is not None
                )
            )
            .all()
        ):
            raise ValueError("IDs must be canonical positive integer strings")
    if (
        not pd.api.types.is_integer_dtype(frame.listening_count)
        or not frame.listening_count.between(1, 2**63 - 1).all()
    ):
        raise ValueError("Counts must be positive int64 integers")
    if frame.duplicated(["user_id", "item_id"]).any():
        raise ValueError("Duplicate user-artist pairs must be resolved before splitting")
    if type(seed) is not int or seed < 0:
        raise ValueError("seed must be a non-negative integer")
    fractions = [validation_fraction, test_fraction]
    if (
        any(type(value) not in (int, float) or not 0 < value < 1 for value in fractions)
        or sum(fractions) >= 1
    ):
        raise ValueError("Holdout fractions must be positive and sum to less than one")

    frame = frame.sort_values(["user_id", "item_id"], key=lambda col: col.map(int)).reset_index(
        drop=True
    )
    # Một bộ sinh ngẫu nhiên với hạt giống cố định (42) để chạy lại ra đúng cách chia.
    rng = random.Random(seed)
    indices: dict[str, list[int]] = {name: [] for name in ["train", "validation", "test"]}
    counts = []
    for user_id, group in frame.groupby("user_id", sort=False):
        n = len(group)
        # Công thức (3.2): n_val = n_test = max(1, ⌊0,1·n⌋), phần còn lại là huấn luyện,
        # nên mỗi người có ít nhất một nghệ sĩ ở mỗi tập.
        n_val = max(1, math.floor(validation_fraction * n))
        n_test = max(1, math.floor(test_fraction * n))
        if n - n_val - n_test < 1:
            raise ValueError(f"User {user_id} has too few interactions for three non-empty splits")
        order = list(group.index)
        rng.shuffle(order)
        indices["validation"].extend(order[:n_val])
        indices["test"].extend(order[n_val : n_val + n_test])
        indices["train"].extend(order[n_val + n_test :])
        counts.append(
            {"user_id": user_id, "train": n - n_val - n_test, "validation": n_val, "test": n_test}
        )
    tables = {
        name: frame.loc[sorted(rows)].reset_index(drop=True) for name, rows in indices.items()
    }
    mappings = {
        "user_id_to_index": {
            value: i for i, value in enumerate(sorted(set(tables["train"].user_id), key=int))
        },
        "item_id_to_index": {
            value: i for i, value in enumerate(sorted(set(tables["train"].item_id), key=int))
        },
    }
    summary: dict[str, Any] = {
        "input": {
            "interactions": len(frame),
            "users": frame.user_id.nunique(),
            "artists": frame.item_id.nunique(),
            "total_listens": sum(map(int, frame.listening_count)),
        },
        "train_catalog_artists": len(mappings["item_id_to_index"]),
        "splits": {},
        "targets": {},
    }
    per_user = pd.DataFrame(counts).set_index("user_id")
    for name, part in tables.items():
        part["user_index"] = part.user_id.map(mappings["user_id_to_index"]).astype("int64")
        part["item_index"] = part.item_id.map(mappings["item_id_to_index"]).astype("Int64")
        summary["splits"][name] = {
            "interactions": len(part),
            "users": part.user_id.nunique(),
            "artists": part.item_id.nunique(),
            "total_listens": sum(map(int, part.listening_count)),
        }
        if name != "train":
            valid = part.item_index.notna()
            available = part.loc[valid].groupby("user_id").size()
            per_user[f"{name}_in_catalog"] = (
                per_user.index.map(available.to_dict()).fillna(0).astype(int)
            )
            per_user[f"{name}_outside_catalog"] = per_user[name] - per_user[f"{name}_in_catalog"]
            summary["targets"][name] = {
                "total": len(part),
                "in_catalog": int(valid.sum()),
                "outside_catalog": int((~valid).sum()),
                "outside_catalog_fraction": float((~valid).mean()),
                "users_with_valid_targets": int((per_user[f"{name}_in_catalog"] > 0).sum()),
                "users_without_valid_targets": int((per_user[f"{name}_in_catalog"] == 0).sum()),
            }
    tables["user_split_counts"] = per_user.reset_index()
    return tables, mappings, summary

In [ ]:
splits, mappings, split_summary = split_interactions(interactions, SEED, VALIDATION_FRACTION, TEST_FRACTION)
names = {"train": "Huấn luyện", "validation": "Xác thực", "test": "Kiểm thử"}
display(pd.DataFrame({names[n]: {
    "Số cặp tương tác": so(s["interactions"]), "Số người dùng": so(s["users"]),
    "Số nghệ sĩ": so(s["artists"]), "Tổng lượt nghe": so(s["total_listens"]),
} for n, s in split_summary["splits"].items()}))
# Bảng 4.1: đích ở xác thực/kiểm thử nằm ngoài danh mục train được báo riêng
display(pd.DataFrame({names[n]: {
    "Tổng số đích": so(t["total"]),
    "Đích ngoài danh mục huấn luyện": f"{so(t['outside_catalog'])} ({so(100 * t['outside_catalog_fraction'], 2)}%)",
    "Số đích hợp lệ": so(t["in_catalog"]),
    "Người dùng có ít nhất một đích hợp lệ": so(t["users_with_valid_targets"]),
    "Người dùng không còn đích": so(t["users_without_valid_targets"]),
} for n, t in split_summary["targets"].items()}))
print("Ánh xạ train:", so(len(mappings["user_id_to_index"])), "người dùng ×",
      so(len(mappings["item_id_to_index"])), "nghệ sĩ")

In [ ]:
for name in ["train", "validation", "test"]:
    print(f"{name}: {so(len(splits[name]))} dòng")
    display(splits[name].head())

In [ ]:
# Một người cụ thể: 50 nghệ sĩ của user 2 được chia 40 / 5 / 5
example = pd.concat([splits[name].assign(tap=name) for name in ["train", "validation", "test"]])
example = example[example.user_id == "2"].sort_values("listening_count", ascending=False)
print(example.tap.value_counts().to_dict())
display(example.head(12))

In [ ]:
# Số tương tác mỗi người ở từng tập
counts = splits["user_split_counts"]
display(counts.head(10))
display(pd.DataFrame({names[n]: {"Nhỏ nhất": so(counts[n].min()), "Trung bình": so(counts[n].mean(), 2),
                                 "Lớn nhất": so(counts[n].max())} for n in names}))
print("Số tương tác train: đúng 40 =", so((counts.train == 40).sum()),
      "| dưới 40 =", so((counts.train < 40).sum()), "| trên 40 =", so((counts.train > 40).sum()))

In [ ]:
# Đích ngoài danh mục: nghệ sĩ chỉ có ở test, item_index rỗng; dòng vẫn được giữ
outside = splits["test"][splits["test"].item_index.isna()]
print(so(len(outside)), "đích test ngoài danh mục train")
display(outside.head())

## 4. Biểu diễn đầu vào

`log2(1 + count)`, điểm đại diện 1–5 theo bốn ngưỡng phân vị 20/40/60/80% **chỉ học trên train**
(công thức 3.3), rồi áp nguyên cho validation/test; thêm dạng nhị phân và bốn ma trận thưa của train.

In [ ]:
REPRESENTATION_CONFIG: dict[str, Any] = {
    "log_base": 2,
    "quantiles": [0.2, 0.4, 0.6, 0.8],
    "quantile_method": "linear",
    "score_boundary": "strictly_greater",
}


def prepare_representations(
    splits: dict[str, pd.DataFrame], mappings: dict[str, dict[str, int]]
) -> tuple[dict[str, pd.DataFrame], dict[str, csr_matrix], dict[str, Any]]:
    """Tạo bốn biểu diễn: lượt nghe gốc, log2(1 + lượt nghe), điểm đại diện 1–5 và nhị phân.

    Nhận ba tập và ánh xạ; trả bảng có thêm cột biểu diễn, bốn ma trận thưa của tập huấn
    luyện và tóm tắt. Ngưỡng điểm 1–5 chỉ học từ tập huấn luyện rồi áp cho cả ba tập
    (chống rò rỉ thông tin từ tập kiểm thử).
    """
    if set(splits) != {"train", "validation", "test"}:
        raise ValueError("Expected train, validation and test splits")
    if set(mappings) != {"user_id_to_index", "item_id_to_index"}:
        raise ValueError("Expected train user and item mappings")
    for mapping in mappings.values():
        if not mapping or any(type(index) is not int for index in mapping.values()):
            raise ValueError("Mappings must contain integer indices")
        if sorted(mapping.values()) != list(range(len(mapping))):
            raise ValueError("Mappings must be contiguous and zero-based")
    for column in ["user_id", "item_id"]:
        if set(splits["train"][column]) != set(mappings[f"{column}_to_index"]):
            raise ValueError("Mappings must contain exactly the train IDs")

    tables = {}
    summary: dict[str, Any] = {"splits": {}}
    columns = ["user_id", "item_id", "listening_count", "user_index", "item_index"]
    for name in ["train", "validation", "test"]:
        frame = splits[name].copy()
        if list(frame.columns) != columns or frame.empty:
            raise ValueError("Expected non-empty Step 4 split columns")
        if frame[["user_id", "item_id", "listening_count", "user_index"]].isna().any().any():
            raise ValueError("Only unknown held-out item indices may be missing")
        if (
            not pd.api.types.is_integer_dtype(frame.listening_count)
            or not frame.listening_count.between(1, 2**63 - 1).all()
        ):
            raise ValueError("Counts must be positive int64 integers")
        if frame.duplicated(["user_id", "item_id"]).any():
            raise ValueError("Duplicate user-artist pairs")
        for column in ["user_id", "item_id"]:
            expected = frame[column].map(mappings[f"{column}_to_index"]).astype("Int64")
            actual = frame[column.replace("_id", "_index")]
            if not pd.api.types.is_integer_dtype(actual) or not actual.astype("Int64").equals(
                expected
            ):
                raise ValueError(f"{name}: indices do not match train mappings")
        if frame.user_index.isna().any() or (name == "train" and frame.item_index.isna().any()):
            raise ValueError("Train indices and all user indices must be known")
        # log2(1 + lượt nghe) làm co lại khoảng giá trị rất lệch của lượt nghe.
        frame["log_count"] = np.log2(1.0 + frame.listening_count.to_numpy(dtype=np.float64))
        # Bốn ngưỡng θ_q là phân vị 20/40/60/80% của tập huấn luyện; chỉ tính khi name == "train".
        if name == "train":
            thresholds = np.quantile(
                frame.log_count, REPRESENTATION_CONFIG["quantiles"], method="linear"
            )
        # Công thức (3.3): r_ui = 1 + số ngưỡng θ_q mà y_ui vượt hẳn (bằng ngưỡng thì giữ mức thấp).
        frame["proxy_score"] = (
            1 + (frame.log_count.to_numpy()[:, None] > thresholds).sum(axis=1)
        ).astype("uint8")
        frame["binary"] = np.ones(len(frame), dtype=np.uint8)
        tables[name] = frame
        summary["splits"][name] = {
            "interactions": len(frame),
            "outside_catalog": int(frame.item_index.isna().sum()),
            "score_counts": {
                str(score): int((frame.proxy_score == score).sum()) for score in range(1, 6)
            },
            "log_count_min": float(frame.log_count.min()),
            "log_count_max": float(frame.log_count.max()),
        }
    train = tables["train"]
    shape = (len(mappings["user_id_to_index"]), len(mappings["item_id_to_index"]))
    coordinates = (train.user_index.to_numpy(), train.item_index.to_numpy(dtype=np.int64))
    matrices = {
        name: csr_matrix((train[column].to_numpy(), coordinates), shape=shape)
        for name, column in {
            "raw_count": "listening_count",
            "log_count": "log_count",
            "proxy_score": "proxy_score",
            "binary": "binary",
        }.items()
    }
    summary["transform"] = {
        **REPRESENTATION_CONFIG,
        "fit_scope": "train only",
        "log_transform": "log2(1 + listening_count)",
        "thresholds": thresholds.tolist(),
        "score_rule": "1 + number of thresholds strictly exceeded; ties stay lower",
        "unobserved": "absent entries; not negative feedback or a one-star score",
        "unknown_items": "retain held-out rows with null item_index; no held-out matrices",
        "matrix_shape": list(shape),
        "matrix_nnz": len(train),
    }
    return tables, matrices, summary

In [ ]:
tables, matrices, rep_summary = prepare_representations(
    {name: splits[name] for name in ["train", "validation", "test"]}, mappings
)
thresholds = rep_summary["transform"]["thresholds"]
print("Bốn ngưỡng θ:", "; ".join(so(t, 4) for t in thresholds), "(6 chữ số:", "; ".join(so(t, 6) for t in thresholds) + ")")
for name, matrix in matrices.items():
    print(f"{name:>12}: {so(matrix.shape[0])} × {so(matrix.shape[1])}, {so(matrix.nnz)} ô quan sát")
display(tables["train"].head())

In [ ]:
# Khoảng lượt nghe ứng với từng mức điểm và số cặp train ở mỗi mức
bounds = [0] + thresholds + [float("inf")]
display(pd.DataFrame({
    "proxy_score": range(1, 6),
    "log2(1+count) trong khoảng": [f"({so(a, 4)}; {so(b, 4)}]" if b != float("inf") else f"> {so(a, 4)}"
                                   for a, b in zip(bounds[:-1], bounds[1:])],
    "lượt nghe xấp xỉ": [f"{so(2 ** a - 1)} – {so(2 ** b - 1)}" if b != float("inf") else f"> {so(2 ** a - 1)}"
                         for a, b in zip(bounds[:-1], bounds[1:])],
    "số cặp train": [so(v) for v in tables["train"].proxy_score.value_counts().sort_index()],
}))

In [ ]:
# Ví dụ: lượt nghe → log2(1 + count) → điểm 1–5; validation dùng nguyên ngưỡng của train
sample = tables["train"].groupby("proxy_score").head(2).sort_values("listening_count")
display(sample[["user_id", "item_id", "listening_count", "log_count", "proxy_score", "binary"]])
display(tables["validation"][["user_id", "item_id", "listening_count", "log_count", "proxy_score"]].head())

# Góc trên bên trái của ma trận điểm đại diện train (0 = chưa nghe)
corner = matrices["proxy_score"][:8, :12].toarray().astype(int)
display(pd.DataFrame(corner, index=[f"user_idx {i}" for i in range(8)], columns=[f"item {j}" for j in range(12)]))

## 5. (Tuỳ chọn) Tải kết quả về máy

In [ ]:
import json
import shutil

out = Path("ket_qua_notebook")
out.mkdir(exist_ok=True)
clean["interactions"].to_parquet(out / "interactions.parquet", index=False)
clean["excluded_interactions"].to_parquet(out / "excluded_interactions.parquet", index=False)
for name in ["train", "validation", "test"]:
    tables[name].to_parquet(out / f"{name}.parquet", index=False)
(out / "id_mappings.json").write_text(json.dumps(mappings, indent=2), encoding="utf-8")
archive = shutil.make_archive("ket_qua_notebook", "zip", out)
try:
    from google.colab import files
    files.download(archive)
except ImportError:
    print("Không chạy trên Colab: tệp nằm ở", archive)